# 03 Lap Time Prediction Model

This notebook trains a machine learning model to predict Formula 1 lap time using race, tyre, driver and team features.

The model is used later to simulate race strategy options and estimate total race time under different pit stop plans.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

try:
    from xgboost import XGBRegressor
    xgboost_available = True
except ImportError:
    xgboost_available = False

os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../visuals", exist_ok=True)

In [ ]:
df = pd.read_csv("../data/raw/synthetic_f1_race_data.csv")

df.head()

## Prepare Modelling Dataset

Safety car laps are removed from model training because they distort normal race pace.

Pit out-laps are also removed to focus the model on representative racing laps.

In [ ]:
model_df = df[(df["safety_car"] == 0) & (df["pit_stop"] == 0)].copy()

model_df.shape

In [ ]:
features = [
    "driver",
    "team",
    "lap",
    "stint",
    "compound",
    "tyre_age",
    "fuel_effect",
    "track_evolution"
]

target = "lap_time_seconds"

X = model_df[features]
y = model_df[target]

categorical_features = ["driver", "team", "compound"]
numeric_features = ["lap", "stint", "tyre_age", "fuel_effect", "track_evolution"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

X_train.shape, X_test.shape

## Train Regression Models

The goal is to predict lap time in seconds.

Mean Absolute Error is especially useful here because it shows the average prediction error in seconds.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        random_state=42
    )
}

if xgboost_available:
    models["XGBoost"] = XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
        objective="reg:squarederror"
    )

In [ ]:
results = []
trained_models = {}

for model_name, model in models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    results.append({
        "Model": model_name,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": rmse,
        "R2 Score": r2_score(y_test, y_pred)
    })

    trained_models[model_name] = pipeline

results_df = pd.DataFrame(results).sort_values(by="MAE")

results_df

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

best_model_name

## Actual vs Predicted Lap Time

In [ ]:
y_pred = best_model.predict(X_test)

predictions = X_test.copy()
predictions["actual_lap_time"] = y_test.values
predictions["predicted_lap_time"] = y_pred
predictions["prediction_error"] = predictions["actual_lap_time"] - predictions["predicted_lap_time"]

predictions.head()

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(predictions["actual_lap_time"], predictions["predicted_lap_time"], alpha=0.7)

min_value = min(predictions["actual_lap_time"].min(), predictions["predicted_lap_time"].min())
max_value = max(predictions["actual_lap_time"].max(), predictions["predicted_lap_time"].max())

plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")

plt.title("Actual vs Predicted Lap Time")
plt.xlabel("Actual Lap Time (seconds)")
plt.ylabel("Predicted Lap Time (seconds)")
plt.tight_layout()
plt.savefig("../visuals/actual_vs_predicted_lap_time.png", bbox_inches="tight")
plt.show()

## Feature Importance

Feature importance helps explain which race factors are most influential in lap time prediction.

In [ ]:
feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()
model_step = best_model.named_steps["model"]

if hasattr(model_step, "feature_importances_"):
    importances = model_step.feature_importances_
else:
    importances = np.zeros(len(feature_names))

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

feature_importance.head(15)

In [ ]:
top_features = feature_importance.head(15).sort_values(by="importance")

plt.figure(figsize=(9, 6))
plt.barh(top_features["feature"], top_features["importance"])

plt.title("Top Features Influencing Lap Time")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig("../visuals/lap_time_feature_importance.png", bbox_inches="tight")
plt.show()

## Save Model Predictions

The predictions are saved for documentation and later analysis.

In [ ]:
predictions.to_csv("../data/processed/lap_time_model_predictions.csv", index=False)

print("Model predictions saved to ../data/processed/lap_time_model_predictions.csv")

## Key Findings

- The model predicts green flag racing lap time using driver, team, tyre and race state features.
- Tyre age, compound, lap number and team pace are expected to strongly influence lap time.
- The model output can be used in the next notebook to compare race strategy scenarios.
- Predicted lap time enables strategy simulation by estimating how much time different pit windows and tyre choices may save or cost.